# Phase 2 — VLM judge on the evaluation images (Colab Pro)

This notebook runs `experiments/exp3_judge.py` over the 3600 images generated in Phase 1, using Qwen2.5-VL-7B-Instruct.

**Requirements:**
- Runtime: A100 or L4 GPU (Qwen-VL-7B in bf16 needs ~16 GB VRAM).
  Set via Runtime → Change runtime type → A100 GPU.
- Google Drive containing the `eval_images/` directory from Phase 1.

**Time estimate:** ~1-2 hours on A100, ~3-4 hours on L4.

## 1. Mount Drive and clone repo

Edit `REPO_URL` and `DRIVE_EVAL_DIR` for your setup.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/SEU-USUARIO/binding-research.git"
BRANCH = "main"
DRIVE_EVAL_DIR = "/content/drive/MyDrive/binding-research/eval_images"

import os, subprocess
REPO_DIR = "/content/binding-research"
if os.path.exists(REPO_DIR):
    print(f'{REPO_DIR} already exists — pulling latest.')
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR

assert os.path.exists(DRIVE_EVAL_DIR), f'Eval images not found at {DRIVE_EVAL_DIR}'
print('Repo and Drive both ready.')

## 2. Install dependencies

Same as Phase 1 — uv for speed.

In [ ]:
!pip install -q uv
!uv pip install -e . --system

## 3. Authenticate with Hugging Face

Qwen2.5-VL is gated by license; you need to accept it once at https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct and have a token.

In [ ]:
from huggingface_hub import login
login()

## 4. Confirm GPU is correct

The cell below prints the GPU name. If it says T4, go back to Runtime → Change runtime type → A100 or L4.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU detected!'
name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name} ({vram_gb:.1f} GB VRAM)')
assert vram_gb >= 20, f'GPU too small ({vram_gb:.1f} GB). Switch to A100 or L4.'

## 5. Smoke test (60 images, ~5 min)

Before judging all 3600, validate the pipeline on a small subset. If this works, you can trust the full run.

In [ ]:
# Symlink the Drive eval dir to where the config expects it.
!mkdir -p data && ln -sfn $DRIVE_EVAL_DIR data/eval_images
!ls -la data/eval_images/manifest.csv

!python experiments/exp3_judge.py \
    --config configs/judge_default.yaml \
    --output-root /content/judgments_smoke \
    --limit 60

In [ ]:
# Inspect the smoke output.
import pandas as pd
smoke = pd.read_csv('/content/judgments_smoke/judgments.csv')
print(f'Smoke rows: {len(smoke)}')
print(f'Binding accuracy: {smoke["binding_correct"].mean():.1%}')
print()
print('Sample of judgments:')
print(smoke[['object', 'color', 'seed', 'object_predicted', 'color_predicted', 'binding_correct']].head(10))

## 6. Full run (3600 images)

Idempotent — if Colab disconnects, just re-run this cell.

In [ ]:
!python experiments/exp3_judge.py \
    --config configs/judge_default.yaml

## 7. Save results to Drive

Critical: copy out before the session ends, or you lose 1-2h of work.

In [ ]:
import shutil
DRIVE_DEST = '/content/drive/MyDrive/binding-research/judgments'
shutil.copytree('/content/binding-research/data/judgments', DRIVE_DEST, dirs_exist_ok=True)
shutil.copytree('/content/binding-research/results',         '/content/drive/MyDrive/binding-research/results', dirs_exist_ok=True)
print(f'Saved to {DRIVE_DEST}')